## LOADING LIBRARIES

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## LOAD DATASET

In [2]:
file = "dataset/student_module_pass_chance_regression.csv"
df = pd.read_csv(file)
df.head()

,Student_ID,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week,Pass_Chance_Percentage
0,ST0001,Female,91.8,79.0,77.5,75.1,7.2,69.8,10.0,2,8.8,76.82
1,ST0002,Male,64.7,44.9,81.3,54.1,7.1,58.8,1.8,5,9.3,35.13
2,ST0003,Female,34.9,37.4,61.0,31.0,13.0,51.8,1.4,6,10.2,25.48
3,ST0004,Female,49.5,44.6,33.6,44.1,1.0,78.1,4.8,4,4.1,32.98
4,ST0005,Male,97.3,92.5,100.0,100.0,4.2,100.0,10.0,1,7.2,91.71


In [3]:
encode = LabelEncoder()
df['Gender'] = encode.fit_transform(df['Gender'])

In [4]:
df.head()

,Student_ID,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week,Pass_Chance_Percentage
0,ST0001,0,91.8,79.0,77.5,75.1,7.2,69.8,10.0,2,8.8,76.82
1,ST0002,1,64.7,44.9,81.3,54.1,7.1,58.8,1.8,5,9.3,35.13
2,ST0003,0,34.9,37.4,61.0,31.0,13.0,51.8,1.4,6,10.2,25.48
3,ST0004,0,49.5,44.6,33.6,44.1,1.0,78.1,4.8,4,4.1,32.98
4,ST0005,1,97.3,92.5,100.0,100.0,4.2,100.0,10.0,1,7.2,91.71


In [5]:
df = df.drop(['Student_ID'], axis=1)

In [6]:
df.isna().sum()

Gender                          0
Attendance_Percentage          45
Assignment_Average             45
Quiz_Average                   45
Midterm_Marks                   0
Study_Hours_Per_Week           45
Previous_Module_Marks          45
Class_Participation_Score       0
Missed_Assignments              0
LMS_Activity_Hours_Per_Week    45
Pass_Chance_Percentage          0
dtype: int64

## DETECT OUTLIERS USING IQR

In [7]:
# Select numerical columns
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns

# Detect outliers using IQR
for column in numerical_columns:
    
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]
    
    print(f"{column}: {len(outliers)} outliers")

Gender: 0 outliers
Attendance_Percentage: 15 outliers
Assignment_Average: 16 outliers
Quiz_Average: 14 outliers
Midterm_Marks: 8 outliers
Study_Hours_Per_Week: 76 outliers
Previous_Module_Marks: 11 outliers
Class_Participation_Score: 0 outliers
Missed_Assignments: 24 outliers
LMS_Activity_Hours_Per_Week: 39 outliers
Pass_Chance_Percentage: 21 outliers


## HANDLE MISSING VALUES USING MEDIAN BCZ ALL FEATURES CONTAIN OUTLIERS

In [8]:
df[numerical_columns] = df[numerical_columns].fillna(
    df[numerical_columns].median()
)

In [9]:
df.isna().sum()

Gender                         0
Attendance_Percentage          0
Assignment_Average             0
Quiz_Average                   0
Midterm_Marks                  0
Study_Hours_Per_Week           0
Previous_Module_Marks          0
Class_Participation_Score      0
Missed_Assignments             0
LMS_Activity_Hours_Per_Week    0
Pass_Chance_Percentage         0
dtype: int64

## REMOVE ALL DETECTED OUTLIERS

In [10]:

print("Dataset before removing outliers:", df.shape)

# Remove outliers using IQR
for column in numerical_columns:
    
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df = df[
        (df[column] >= lower_bound) &
        (df[column] <= upper_bound)
    ]

print("Dataset after removing outliers:", df.shape)
df.reset_index(drop=True, inplace=True)

df.head()

Dataset before removing outliers: (3000, 11)
Dataset after removing outliers: (2791, 11)


,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week,Pass_Chance_Percentage
0,0,91.8,79.0,77.5,75.1,7.2,69.8,10.0,2,8.8,76.82
1,1,64.7,44.9,81.3,54.1,7.1,58.8,1.8,5,9.3,35.13
2,1,97.3,92.5,100.0,100.0,4.2,100.0,10.0,1,7.2,91.71
3,0,70.0,72.9,89.6,78.9,8.8,60.8,8.2,5,6.1,57.93
4,0,64.7,75.7,55.9,85.0,3.7,44.4,6.8,2,7.7,63.29


## Feature splitting

In [11]:
X = df.drop(['Pass_Chance_Percentage'], axis=1)
y = df['Pass_Chance_Percentage']
print(X)
print(y)

      Gender  Attendance_Percentage  Assignment_Average  Quiz_Average  \
0          0                   91.8                79.0          77.5   
1          1                   64.7                44.9          81.3   
2          1                   97.3                92.5         100.0   
3          0                   70.0                72.9          89.6   
4          0                   64.7                75.7          55.9   
...      ...                    ...                 ...           ...   
2786       1                   84.9                72.1          71.4   
2787       0                   78.6                63.7          59.2   
2788       1                   83.9                88.2          91.2   
2789       0                   65.6                47.7          52.7   
2790       1                   72.3                56.6          59.8   

      Midterm_Marks  Study_Hours_Per_Week  Previous_Module_Marks  \
0              75.1                   7.2              

## SPLIT DATASET

In [12]:
X_train, X_test, y_train,y_test=train_test_split(X, y, test_size=.2, random_state=42)
X_train.shape

(2232, 10)

In [13]:
X_test.shape

(559, 10)

## Scale our data to prevent data leakage during model testing

In [14]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train

,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week
1644,1,92.2,72.6,89.4,85.3,13.0,87.4,4.6,2,10.9
2389,1,68.6,69.4,71.3,60.6,8.4,80.2,4.9,4,9.9
1807,1,69.3,79.2,69.3,66.6,14.8,47.0,5.4,2,13.6
100,0,62.7,47.4,45.5,59.7,8.6,69.7,7.1,1,13.0
315,0,92.2,84.7,83.0,81.7,5.2,79.9,7.9,2,7.1
...,...,...,...,...,...,...,...,...,...,...
1638,0,80.0,58.1,79.6,53.8,6.7,72.4,3.5,0,5.7
1095,0,78.6,54.7,55.3,41.4,9.7,62.1,8.4,2,11.5
1130,1,60.9,100.0,79.9,81.6,12.2,69.7,4.8,2,10.8
1294,0,85.2,59.3,88.9,78.2,12.8,78.6,9.4,1,11.1


In [15]:
X_test

,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week
2764,1,72.4,73.2,80.4,70.0,7.7,72.1,5.5,0,5.8
2270,0,75.8,47.0,56.9,69.1,11.7,67.8,4.9,0,14.2
1993,0,100.0,63.7,75.2,71.5,3.8,64.4,8.9,2,11.1
2462,0,74.8,77.5,77.9,96.0,9.5,55.8,8.1,2,15.1
2130,0,89.9,60.9,70.6,55.8,5.0,86.4,8.4,1,7.1
...,...,...,...,...,...,...,...,...,...,...
2518,0,93.3,89.2,61.9,89.6,8.4,76.4,9.4,0,9.3
157,0,60.8,60.8,83.5,69.1,7.9,52.3,2.1,6,9.8
2566,0,96.6,73.6,85.3,80.3,14.8,65.1,9.2,1,14.0
2003,0,72.7,54.0,50.3,72.5,7.2,73.6,5.8,3,8.5


## Model selection

## LINEAR REGRESSION

In [16]:
model_lr = LinearRegression()
model_lr.fit(X_train_scaled, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](10,)","[ -0.14, 13.32, 10.62,..., 9.45,-11.3 , 5.12]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,27.18
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,10
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(10)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](10,)","[23.6 ,16.89,12.41,..., 7.01, 6.68, 4.86]"


In [17]:
y_lr_pred = model_lr.predict(X_test_scaled)
# y_lr_pred

In [18]:
y_test

2764    61.73
2270    51.88
1993    64.63
2462    68.55
2130    55.49
        ...  
2518    80.04
157     45.61
2566    86.91
2003    53.56
238     51.06
Name: Pass_Chance_Percentage, Length: 559, dtype: float64

In [19]:
print("Linear Regression Results:")
print(f"R^2 Score: {r2_score(y_test, y_lr_pred):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_lr_pred):.4f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_lr_pred):.4f}")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, y_lr_pred)):.4f}")

Linear Regression Results:
R^2 Score: 0.7690
Mean Absolute Error: 4.3364
Mean Squared Error: 29.7473
Root Mean Squared Error: 5.4541


In [20]:
print("Coefficients:", model_lr.coef_)

Coefficients: [ -0.1389843   13.32336127  10.62015864   6.00248531  11.46145513
  12.67583639   5.73726941   9.45188958 -11.29665324   5.11738436]


In [21]:
print("Intercept:", model_lr.intercept_)

Intercept: 27.18125949292351


In [22]:
# New student samples
new_students = pd.DataFrame({
    "Gender": [0, 1, 0, 1, 0, 1],
    "Attendance_Percentage": [95, 80, 65, 45, 88, 5],
    "Assignment_Average": [90, 75, 60, 40, 85, 5],
    "Quiz_Average": [92, 72, 58, 35, 82, 60],
    "Midterm_Marks": [88, 78, 55, 38, 84, 65],
    "Study_Hours_Per_Week": [18, 12, 7, 3, 15, 1],
    "Previous_Module_Marks": [86, 74, 58, 42, 80, 5],
    "Class_Participation_Score": [9, 7, 5, 3, 8, 6],
    "Missed_Assignments": [0, 1, 3, 7, 1, 2],
    "LMS_Activity_Hours_Per_Week": [16, 11, 6, 2, 14, 9]
})

new_students



,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week
0,0,95,90,92,88,18,86,9,0,16
1,1,80,75,72,78,12,74,7,1,11
2,0,65,60,58,55,7,58,5,3,6
3,1,45,40,35,38,3,42,3,7,2
4,0,88,85,82,84,15,80,8,1,14
5,1,5,5,60,65,1,5,6,2,9


In [23]:
new_students_scaled = scaler.transform(new_students)

new_students_scaled

array([[ 0.        ,  0.91896272,  0.85294118,  0.88148148,  0.8136646 ,
         0.86868687,  0.80609418,  0.88888889,  0.        ,  0.82901554],
       [ 1.        ,  0.67585089,  0.63235294,  0.58518519,  0.65838509,
         0.56565657,  0.6398892 ,  0.66666667,  0.16666667,  0.56994819],
       [ 0.        ,  0.43273906,  0.41176471,  0.37777778,  0.30124224,
         0.31313131,  0.41828255,  0.44444444,  0.5       ,  0.31088083],
       [ 1.        ,  0.10858995,  0.11764706,  0.03703704,  0.03726708,
         0.11111111,  0.1966759 ,  0.22222222,  1.16666667,  0.10362694],
       [ 0.        ,  0.80551053,  0.77941176,  0.73333333,  0.7515528 ,
         0.71717172,  0.72299169,  0.77777778,  0.16666667,  0.7253886 ],
       [ 1.        , -0.53970827, -0.39705882,  0.40740741,  0.45652174,
         0.01010101, -0.31578947,  0.55555556,  0.33333333,  0.46632124]])

In [24]:
new_predictions = model_lr.predict(new_students_scaled)

new_predictions

array([91.38034528, 71.99770093, 49.55249561, 22.37603694, 81.62609102,
       25.50069289])

## random forest

In [25]:
model = RandomForestRegressor()
model.fit(X_train_scaled, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the numb

In [26]:
y_pred = model.predict(X_test_scaled)
y_pred

array([63.1675, 62.8111, 66.9919, 70.3999, 67.2347, 56.9827, 76.1096,
       47.7794, 60.53  , 74.8806, 58.6126, 68.5038, 61.8497, 72.7308,
       63.9291, 71.8168, 67.4142, 80.0763, 68.4498, 67.2909, 74.6233,
       47.8681, 70.6851, 68.1505, 56.9177, 52.8749, 74.9658, 77.5515,
       61.6053, 54.2056, 62.8978, 80.6318, 65.4763, 54.2514, 67.3735,
       63.0108, 73.5387, 66.861 , 64.0015, 57.627 , 82.2167, 80.2502,
       65.0239, 58.9309, 61.4465, 76.351 , 72.0349, 72.2383, 77.7787,
       53.7019, 43.4105, 63.8094, 54.9732, 44.9118, 62.8223, 64.4499,
       61.7205, 65.9239, 63.4502, 53.7287, 59.1467, 47.6251, 75.1501,
       66.1588, 85.6059, 66.2223, 66.8345, 61.2518, 70.0832, 63.8663,
       72.4459, 64.9904, 80.025 , 46.9836, 74.13  , 76.1323, 63.637 ,
       61.7169, 69.5482, 77.1429, 74.408 , 60.1853, 57.2917, 52.9352,
       53.6192, 65.6195, 80.8233, 69.76  , 64.6657, 55.8738, 53.5775,
       63.1914, 64.4754, 64.2382, 73.0495, 75.6685, 44.565 , 75.7211,
       80.6759, 73.6

In [27]:
y_test

2764    61.73
2270    51.88
1993    64.63
2462    68.55
2130    55.49
        ...  
2518    80.04
157     45.61
2566    86.91
2003    53.56
238     51.06
Name: Pass_Chance_Percentage, Length: 559, dtype: float64

In [28]:
print(mean_absolute_error(y_test, y_pred))
print(mean_squared_error(y_test, y_pred))
print(r2_score(y_test, y_pred))

4.8007084078711975
35.071467427620746
0.727705998837592


In [29]:
X_test.columns

Index(['Gender', 'Attendance_Percentage', 'Assignment_Average', 'Quiz_Average',
       'Midterm_Marks', 'Study_Hours_Per_Week', 'Previous_Module_Marks',
       'Class_Participation_Score', 'Missed_Assignments',
       'LMS_Activity_Hours_Per_Week'],
      dtype='str')

## Testing no new sample

In [30]:
# New student samples
new_students = pd.DataFrame({
    "Gender": [0, 1, 0, 1, 0],
    "Attendance_Percentage": [95, 80, 65, 45, 88],
    "Assignment_Average": [90, 75, 60, 40, 85],
    "Quiz_Average": [92, 72, 58, 35, 82],
    "Midterm_Marks": [88, 78, 55, 38, 84],
    "Study_Hours_Per_Week": [18, 12, 7, 3, 15],
    "Previous_Module_Marks": [86, 74, 58, 42, 80],
    "Class_Participation_Score": [9, 7, 5, 3, 8],
    "Missed_Assignments": [0, 1, 3, 7, 1],
    "LMS_Activity_Hours_Per_Week": [16, 11, 6, 2, 14]
})

new_students

,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week
0,0,95,90,92,88,18,86,9,0,16
1,1,80,75,72,78,12,74,7,1,11
2,0,65,60,58,55,7,58,5,3,6
3,1,45,40,35,38,3,42,3,7,2
4,0,88,85,82,84,15,80,8,1,14


In [31]:
new_students_scaled = scaler.transform(new_students)

new_students_scaled

array([[0.        , 0.91896272, 0.85294118, 0.88148148, 0.8136646 ,
        0.86868687, 0.80609418, 0.88888889, 0.        , 0.82901554],
       [1.        , 0.67585089, 0.63235294, 0.58518519, 0.65838509,
        0.56565657, 0.6398892 , 0.66666667, 0.16666667, 0.56994819],
       [0.        , 0.43273906, 0.41176471, 0.37777778, 0.30124224,
        0.31313131, 0.41828255, 0.44444444, 0.5       , 0.31088083],
       [1.        , 0.10858995, 0.11764706, 0.03703704, 0.03726708,
        0.11111111, 0.1966759 , 0.22222222, 1.16666667, 0.10362694],
       [0.        , 0.80551053, 0.77941176, 0.73333333, 0.7515528 ,
        0.71717172, 0.72299169, 0.77777778, 0.16666667, 0.7253886 ]])

In [32]:
new_students_scaled = pd.DataFrame(
    scaler.transform(new_students),
    columns=new_students.columns
)

new_students_scaled

,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week
0,0.0,0.918963,0.852941,0.881481,0.813665,0.868687,0.806094,0.888889,0.000000,0.829016
1,1.0,0.675851,0.632353,0.585185,0.658385,0.565657,0.639889,0.666667,0.166667,0.569948
2,0.0,0.432739,0.411765,0.377778,0.301242,0.313131,0.418283,0.444444,0.500000,0.310881
3,1.0,0.108590,0.117647,0.037037,0.037267,0.111111,0.196676,0.222222,1.166667,0.103627
4,0.0,0.805511,0.779412,0.733333,0.751553,0.717172,0.722992,0.777778,0.166667,0.725389


In [33]:
new_predictions = model.predict(new_students_scaled)

new_predictions

array([88.616 , 71.4621, 50.6174, 39.2539, 79.6205])